### Creating new variables/columns into the dataset WITH DFS

In [7]:
import pandas as pd
import numpy as np

# pip install featuretools
import featuretools as ft

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.model_selection import train_test_split

# load data
df = pd.read_csv("HeartDisease_Cleaned.csv")

df

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST
0,40,1,140,289,0,172,0,0.0,3,0,0,1,0,0,0,1,0
1,49,0,160,180,0,156,0,1.0,2,1,0,0,1,0,0,1,0
2,37,1,130,283,0,98,0,0.0,3,0,0,1,0,0,0,0,1
3,48,0,138,214,0,108,1,1.5,2,1,1,0,0,0,0,1,0
4,54,1,150,195,0,122,0,0.0,3,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,110,264,0,132,0,1.2,2,1,0,0,0,1,0,1,0
914,68,1,144,193,1,141,0,3.4,2,1,1,0,0,0,0,1,0
915,57,1,130,131,0,115,1,1.2,2,1,1,0,0,0,0,1,0
916,57,0,130,236,0,174,0,0.0,2,1,0,1,0,0,1,0,0


**DFS - Deep Feature Synthesis - automate generation of new columns via BRUTE FORCE**

By default, you should drop the target variable, because you can't have the target variable to be in any part of any new variable (because that is target leakage => not realistic from an actual ML model point of view).

In [8]:
# define the target variable
target = "HeartDisease"

# save for later use
y = df[target]

# create an entity set with the red wine data
es = ft.EntitySet(id="heart_disease")
es = es.add_dataframe(dataframe_name="heart_disease_data", 
                      dataframe=df.drop(target, axis=1),
                      index="index",
                      make_index=True)


# Use DFS to generate new variables (features), aggregate BASED on target variable: quality
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="heart_disease_data",
    
    # agg_primitives=[],
    # if you use agg primitives, define also the calculations:
    # you can also use groupby trans primitives to nominal categories
    # groupby_trans_primitives=["cum_mean"],
    # trans_primitives=["multiply_numeric"],
    trans_primitives=["divide_numeric", "multiply_numeric"],
    # increase depth if you want to capture more complex
    # relationships in the data
    max_depth=1
)

# division a bit problematic, since often times
# values we divide with something very small (but not 0)
# this results in many rows and columns to go into infinity
# we have to take care of this

# THIS PART ONLY REQUIRED IF YOU DO DIVISION
# IT REMOVES ALL COLUMNS WITH INFINITE VALUES
feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.dropna(axis=1, inplace=True)

# re-attach y back to feature_matrix
feature_matrix[target] = y


**After this, use RFE or something similar to see which variables to keep and which to drop!**

In [9]:
# typical X/y -split
X = feature_matrix.drop(target, axis=1)
y = feature_matrix[target]

# define model (random forest classifier in this example, because the target is 0 or 1)
# technically you can use pretty much any classic ML algorithm
# (random forest does not need scaled values, which keeps this example simple)
model = RandomForestClassifier(random_state=42)

# NOTE! LogisticRegression is faster, but it needs scaled values to converge
# model = LogisticRegression(max_iter=1000)

# create RFE, place the model and choose amount of optimal variables
rfe = RFE(estimator=model, n_features_to_select=12)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and the results
rankings = rfe.ranking_
support = rfe.support_

results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# if having lots of new variables, use Data Wrangler or
# something else to see all results
results_df.iloc[0:15]

,Feature,Ranking,Selected
8,ST_Slope,1,True
17,Age / ST_Slope,1,True
19,ChestPainType_ASY / MaxHR,1,True
56,ST_Slope / Age,1,True
59,Sex / MaxHR,1,True
60,Sex / ST_Slope,1,True
42,Oldpeak / MaxHR,1,True
46,RestingBP / ST_Slope,1,True
169,RestingBP * ST_Slope,1,True
158,MaxHR * ST_Slope,1,True


In [10]:
# Correlation with the target: are new features better than originals? 
from scipy.stats import spearmanr
import pandas as pd

corr = X.apply(lambda s: spearmanr(s, y)[0]).abs().sort_values(ascending=False)
corr_df = pd.DataFrame({"abs_spearman": corr,
                        "is_new": [(" / " in c) or (" * " in c) for c in corr.index]})
print(corr_df.head(15))
print("Best original:", corr_df[~corr_df.is_new].abs_spearman.max().round(3))
print("Best new:     ", corr_df[corr_df.is_new].abs_spearman.max().round(3))

                              abs_spearman  is_new
MaxHR * ST_Slope                  0.594736    True
ST_Slope                          0.591913   False
ChestPainType_ASY / ST_Slope      0.577649    True
Age / ST_Slope                    0.561556    True
ST_Slope / Age                    0.561556    True
Sex / ST_Slope                    0.560985    True
RestingBP / ST_Slope              0.547325    True
ChestPainType_ASY / MaxHR         0.522907    True
ChestPainType_ASY                 0.516716   False
ChestPainType_ASY * Sex           0.515213    True
ExerciseAngina / ST_Slope         0.513009    True
Age * ChestPainType_ASY           0.511056    True
ExerciseAngina / MaxHR            0.503928    True
ExerciseAngina                    0.494282   False
ExerciseAngina * Sex              0.491511    True
Best original: 0.592
Best new:      0.595


C:\Users\Gayani\AppData\Local\Temp\ipykernel_30324\951616944.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr = X.apply(lambda s: spearmanr(s, y)[0]).abs().sort_values(ascending=False)


In [11]:
# pip install statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import pandas as pd
# perform X/y -split
# if you  have more than one independent variable, list them all here
# leave out the target variable! (dependent variable)

# this is a nice and common trick => everything EXCEPT target variable => support variable
X = df.drop("HeartDisease", axis=1)

# have only the target variable here (dependent variable)
y = df["HeartDisease"]



# VIF dataframe 
# VIF = Variance Inflation Factor
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

# variables with high VIF-value 
# can mean multlicollinearity (variables providing same linear
# relationships in the data, potentially confusing the ML algorithm
# this might be good info when deciding if some variable needs to be removed
vif_data.sort_values(by="VIF", ascending=False)

C:\Users\Gayani\AppData\Local\Temp\ipykernel_30324\3752187498.py:22: UserWarning: The design matrix is poorly conditioned (condition number=5.88e+15). VIF calculations may be numerically unstable.
  vif_data["VIF"] = [variance_inflation_factor(X.values, i)
c:\Users\Gayani\Advanced Data Analytics\AdvancedDataAnalytics\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Gayani\Advanced Data Analytics\AdvancedDataAnalytics\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Gayani\Advanced Data Analytics\AdvancedDataAnalytics\.venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank

,feature,VIF
12,ChestPainType_TA,1.000800e+15
15,RestingECG_ST,1.000800e+15
13,RestingECG_LVH,1.000800e+15
14,RestingECG_Normal,1.000800e+15
10,ChestPainType_ATA,1.000800e+15
11,ChestPainType_NAP,1.000800e+15
9,ChestPainType_ASY,1.000800e+15
8,ST_Slope,1.624709e+00
6,ExerciseAngina,1.562173e+00
5,MaxHR,1.529045e+00


In [14]:
# the 12 features selected by RFE (from the results table)
selected = ["ST_Slope",
            "Age / ST_Slope",
            "ChestPainType_ASY / MaxHR",
            "ST_Slope / Age",
            "Sex / MaxHR",
            "Sex / ST_Slope",
            "Oldpeak / MaxHR",
            "RestingBP / ST_Slope",
            "RestingBP * ST_Slope",
            "MaxHR * ST_Slope",
            "Cholesterol * RestingBP",
            "Cholesterol * MaxHR"]

X_new = feature_matrix[selected]

# VIF dataframe
# VIF = Variance Inflation Factor
vif_data = pd.DataFrame()
vif_data["feature"] = X_new.columns

# calculating VIF for each feature
vif_data["VIF"] = [variance_inflation_factor(X_new.values, i)
                   for i in range(len(X_new.columns))]

vif_data.sort_values(by="VIF", ascending=False)

,feature,VIF
0,ST_Slope,31.198248
11,Cholesterol * MaxHR,19.265008
10,Cholesterol * RestingBP,16.523985
9,MaxHR * ST_Slope,15.825890
7,RestingBP / ST_Slope,15.468589
1,Age / ST_Slope,14.262872
8,RestingBP * ST_Slope,12.734133
5,Sex / ST_Slope,10.257725
4,Sex / MaxHR,8.123988
3,ST_Slope / Age,7.795409


The original features had VIF values of 1.1 to 1.6, so there was no multicollinearity. After DFS, 8 of the 12 RFE-selected features had VIF above 10 (ST_Slope reached 31.2), because most of them were built from the same variables, mainly ST_Slope. The interaction features therefore add redundancy rather than new information. Together with the lower accuracy after RFE (0.8060 vs 0.8289), this suggests the new features are not useful. Only features that stay below VIF 10 and improve cross-validated accuracy should be kept.